# Feature Engineering for Community Area Aggregation
This notebook works as the other Feature Engineering Notebooks except it uses the dataset aggregated on Community Area

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import holidays
from sklearn.metrics.pairwise import haversine_distances
import h3

In [ ]:
# Load all complete grids from the data folder
data_folder = Path("../data/processed/")
grid_files = list(data_folder.glob("complete_grid_*.parquet"))

# h3_8 was dropped from the prediction scope (see SPATIAL_LEVELS in 3.2 Prediction_CommonGround):
# we predict at three spatial levels (h3_7, census, community) only. Skip any lingering res-8 grid so
# it is neither engineered nor split into prediction_split/h3_8/. res-8 stays a descriptive level in
# 1.3/1.4/2.x; it is simply no longer a prediction unit.
grid_files = [f for f in grid_files if "h3_8" not in f.stem]
print(f"Found {len(grid_files)} complete grid files.")

# load each grid file into own data frame and save them in a dictionary
grid_dfs = {}
for grid_file in grid_files:
    grid_name = grid_file.stem  # Get the file name without extension
    grid_dfs[grid_name] = pd.read_parquet(grid_file)
    print(f"Loaded {grid_name} with shape: {grid_dfs[grid_name].shape}")

In [ ]:
# The grids key tracts/community areas as STRINGS ('17031010100', '8'). Left to itself read_csv
# would parse those columns as int64 and the merge would match nothing, silently producing all-NaN
# POI columns -- so force the key dtypes.
poi_keys = {"geoid10": str, "commarea": str, "h3_index": str}

pois_cat_wide_7 = pd.read_csv("../data/processed/chicago_pois_category_wide_7.csv", dtype=poi_keys)
pois_cat_wide_census = pd.read_csv("../data/processed/chicago_pois_category_wide_census.csv", dtype=poi_keys)
pois_cat_wide_comm = pd.read_csv("../data/processed/chicago_pois_category_wide_community.csv", dtype=poi_keys)
weather_df = pd.read_parquet("../data/weather_data.parquet")

for name, df in [("pois_7", pois_cat_wide_7),
                 ("pois_census", pois_cat_wide_census), ("pois_community", pois_cat_wide_comm)]:
    print(f"Loaded {name:15s} shape: {str(df.shape):10s} key: {df.columns[0]}")
print(f"Loaded weather data with shape: {weather_df.shape}")

In [4]:
for grid_name, grid_df in grid_dfs.items():
    print(f"Columns names in {grid_name}:")
    print(grid_df.columns)

Columns names in complete_grid_community_daily:
Index(['timestamp', 'commarea', 'Total_Trip_Start', 'Unique Taxis',
       'AvgTripSeconds', 'AvgTripMiles', 'AvgFare', 'MostCommonCompany',
       'CompanyCount', 'PickupLongitude', 'PickupLatitude', 'Total_Trip_End',
       'lat', 'lon'],
      dtype='str')
Columns names in complete_grid_h3_7_hourly:
Index(['timestamp', 'h3_index_7', 'Total_Trip_Start', 'Unique Taxis',
       'AvgTripSeconds', 'AvgTripMiles', 'AvgFare', 'MostCommonCompany',
       'CompanyCount', 'PickupLongitude', 'PickupLatitude', 'Total_Trip_End',
       'lat', 'lon'],
      dtype='str')
Columns names in complete_grid_h3_8_daily:
Index(['timestamp', 'h3_index_8', 'Total_Trip_Start', 'Unique Taxis',
       'AvgTripSeconds', 'AvgTripMiles', 'AvgFare', 'MostCommonCompany',
       'CompanyCount', 'PickupLongitude', 'PickupLatitude', 'Total_Trip_End',
       'lat', 'lon'],
      dtype='str')
Columns names in complete_grid_h3_8_hourly:
Index(['timestamp', 'h3_index_8', 'To

In [5]:
# Find Null Values and add percentage of NaN values for each column
# Only print columns with NaN values
def generate_nan_report(df):
    is_na_df = df.isna().sum()
    is_na_df = is_na_df[is_na_df > 0]
    is_na_df = pd.DataFrame(is_na_df, columns=['NaN Count'])
    is_na_df['Total Count'] = len(df)
    nan_percentage = (is_na_df['NaN Count'] / is_na_df['Total Count']) * 100
    # Format the NaN percentage as a string with 2 decimal places and add a percentage sign
    is_na_df['NaN Percentage'] = nan_percentage.map(lambda x: f"{x:.2f}%")
    return is_na_df

# Check for any NaN values in all columns
print("Only showing columns with NaN values:")
for grid_name, grid_df in grid_dfs.items():
    nan_report = generate_nan_report(grid_df)
    if not nan_report.empty:
        print(f"NaN Report for {grid_name}:")
        print(nan_report)
        print(40 * "-")
    else:
        print(f"No NaN values found in {grid_name}.")
        print(40 * "-")

Only showing columns with NaN values:
NaN Report for complete_grid_community_daily:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds        474        66451          0.71%
AvgTripMiles          474        66451          0.71%
AvgFare               474        66451          0.71%
----------------------------------------
NaN Report for complete_grid_h3_7_hourly:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds    2298278      2464728         93.25%
AvgTripMiles      2298278      2464728         93.25%
AvgFare           2298278      2464728         93.25%
----------------------------------------
NaN Report for complete_grid_h3_8_daily:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds     336134       380583         88.32%
AvgTripMiles       336134       380583         88.32%
AvgFare            336134       380583         88.32%
----------------------------------------


NaN Report for complete_grid_h3_8_hourly:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds    8824834      9133992         96.62%
AvgTripMiles      8824834      9133992         96.62%
AvgFare           8824834      9133992         96.62%
----------------------------------------
NaN Report for complete_grid_community_hourly:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds     688356      1594824         43.16%
AvgTripMiles       688356      1594824         43.16%
AvgFare            688356      1594824         43.16%
----------------------------------------
NaN Report for complete_grid_h3_7_daily:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds      81058       102697         78.93%
AvgTripMiles        81058       102697         78.93%
AvgFare             81058       102697         78.93%
----------------------------------------


NaN Report for complete_grid_census_hourly:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds   12022051     12427200         96.74%
AvgTripMiles     12022051     12427200         96.74%
AvgFare          12022051     12427200         96.74%
----------------------------------------
NaN Report for complete_grid_census_daily:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds     460058       517800         88.85%
AvgTripMiles       460058       517800         88.85%
AvgFare            460058       517800         88.85%
----------------------------------------


In [6]:
# Columns aggregated from the SAME trips as the target, Total_Trip_Start. None is knowable at
# prediction time, and each leaks the target almost perfectly (measured on complete_grid_h3_8_hourly):
#
#   Unique Taxis       > 0 iff demand > 0 (100.00% of rows); corr 0.995; numerically IDENTICAL to the
#                      target in 64% of rows
#   CompanyCount       > 0 iff demand > 0 (100.00%); corr 0.843
#   MostCommonCompany  != "" iff demand > 0 (100.00%)
#   AvgFare            notna() iff demand > 0 (100.00%) -- the MISSINGNESS alone is the target, so no
#   AvgTripMiles       imputation can rescue these: the NaN pattern still gives the answer away
#   AvgTripSeconds
#   Total_Trip_End     contemporaneous dropoffs; > 0 iff demand > 0 (97.8%); corr 0.780
#   PickupLatitude     trip-average coordinates, filled with the cell centroid where there were no
#   PickupLongitude    trips -- so they equal lat/lon iff the cell is empty (99.97%). Redundant with
#                      lat/lon, which give the same location with no leak.
#
# Dropped in 3.1 rather than in 1.4 so the grids stay a complete descriptive dataset: a LAGGED
# version of these (dropoffs at t-1, average fare at t-24h) is a legitimate, non-leaking predictor
# and is still one step away.
#
# Side effect: these are the only columns that carry NaN, so the modelling data comes out clean.
#
# The drop itself happens inside engineer_features() below, which runs on the frame freshly loaded
# from parquet in the pipeline loop. Doing it here over grid_dfs would have no effect: .drop()
# returns a new frame rather than mutating in place, and the pipeline never reads grid_dfs anyway.
LEAKY_COLUMNS = [
    'Unique Taxis', 'CompanyCount', 'MostCommonCompany',
    'AvgFare', 'AvgTripMiles', 'AvgTripSeconds',
    'Total_Trip_End',
    'PickupLatitude', 'PickupLongitude',
]

## 1. Spatial Features

In [7]:
# Calculate distance to Chicago Loop (downtown center)
# The Haversine distance is the shortest route over the earth's surface
# (distance between two points on a sphere, calculated using their latitudes and longitudes)
def new_haversine_distance_to_loop(lat1, lon1, ):
    # Use scikit-learn's haversine_distances function
    lat2=41.8781 # Chicago Loop latitude
    lon2=-87.6298 # Chicago Loop longitude
    # Convert latitudes and longitudes from degrees to radians
    coords_1 = np.radians(np.column_stack((lat1, lon1)))
    coords_2 = np.radians(np.array([[lat2, lon2]]))
    # Convert from radians to kilometers
    distances = haversine_distances(coords_1, coords_2) * 6371  
    return distances.flatten()

## 2. Time Features

In [8]:
def add_holiday_feature(grid_df, temporal_column):
    dates = grid_df[temporal_column].dt.normalize()
    us_holidays = holidays.USA(years=dates.dt.year.unique(), state='IL')
    holiday_dates = pd.DatetimeIndex(sorted(us_holidays.keys()))

    grid_df['is_holiday'] = dates.isin(holiday_dates).astype(int)

    # "Near a holiday" means the calendar day before or after one. Computing it with .shift(1) on the
    # frame would step to the neighbouring ROW -- which is the next spatial unit in the same hour,
    # not the next day -- so build the neighbouring dates explicitly instead.
    day = pd.Timedelta(days=1)
    near = holiday_dates.union(holiday_dates - day).union(holiday_dates + day)
    grid_df['is_near_holiday'] = dates.isin(near).astype(int)
    return grid_df


def temporal_feature_engineering(grid_df, temporal_column, temporal_level):
    # Add temporal features.
    # The hour features only exist for the hourly grids -- a daily grid is floored to midnight, so
    # hour_of_day would be a constant 0. (This used to be gated on `temporal_column == 'hour'`, but
    # the column is called 'timestamp', so the hour features were never created at all.)

    # Calendar features
    if temporal_level == 'hourly':
        grid_df['hour_of_day'] = grid_df[temporal_column].dt.hour
    grid_df['month'] = grid_df[temporal_column].dt.month
    grid_df['day_of_week'] = grid_df[temporal_column].dt.weekday
    grid_df['is_weekend'] = (grid_df['day_of_week'] >= 5).astype(int)
    grid_df['weekday'] = grid_df[temporal_column].dt.weekday
    grid_df["season"] = grid_df[temporal_column].dt.month % 12 // 3 + 1 # Mapping: 1 = winter, 2 = spring, 3 = summer, 4 = autumn

    # Cyclic temporal features
    if temporal_level == 'hourly':
        grid_df['hour_sin'] = np.sin(2 * np.pi * grid_df['hour_of_day'] / 24)
        grid_df['hour_cos'] = np.cos(2 * np.pi * grid_df['hour_of_day'] / 24)
    grid_df['month_sin'] = np.sin(2 * np.pi * grid_df['month'] / 12)
    grid_df['month_cos'] = np.cos(2 * np.pi * grid_df['month'] / 12)
    grid_df["season_sin"] = np.sin(2 * np.pi * grid_df["season"] / 4)
    grid_df["season_cos"] = np.cos(2 * np.pi * grid_df["season"] / 4)
    return grid_df


# Historical base demand — computed ONLY from train (no leakage)
def add_base_demand(df, base, keys, global_base):
    df = df.merge(base, on=keys, how="left")
    # unseen keys -> global mean
    df["base_demand"] = df["base_demand"].fillna(global_base)  
    return df


def base_demand_keys(spatial_column, temporal_level):
    """The profile a cell's historical demand is averaged over, per temporal level."""
    if temporal_level == 'hourly':
        return [spatial_column, 'hour_of_day', 'is_weekend']
    # A daily grid has no hour; day_of_week already carries the weekday/weekend distinction.
    return [spatial_column, 'day_of_week']

## 3. Weather Features

In [9]:
def create_weather_features(grid_df, weather_df, temporal_column, time_step_column="time_step"):
    # Work on a copy: the raw wind columns get derived features, and mutating the caller's frame
    # would re-derive them on every grid.
    weather_df = weather_df.copy()

    # combine the raw wind data into the wind speed and direction
    weather_df["wind_speed"] = (weather_df["u10"]**2 + weather_df["v10"]**2) ** 0.5
    weather_df["wind_dir"] = (270 - np.degrees(np.arctan2(weather_df["u10"], weather_df["v10"]))*180/np.pi)%360

    weather_columns = [c for c in weather_df.columns if c != time_step_column]

    # join weather and taxi data upon the time as key 
    grid_df = grid_df.merge(
        weather_df,
        left_on=temporal_column,
        right_on=time_step_column,
        how="left"
    )

    # 1.4 now cuts the taxi grid at the last hour the weather series covers, and the series has no
    # internal gaps, so every row must find its weather. Assert that instead of filling.
    #
    # This replaces a blanket grid_df.ffill(). That fill was wrong twice over: it froze the last
    # observed weather across the uncovered tail (which, under a temporal split, sat entirely in the
    # TEST set), and it also walked into every other column -- and since consecutive rows are
    # DIFFERENT spatial units in the same hour, it copied one cell's values into its neighbour rather
    # than filling forward in time at all.
    missing = int(grid_df[weather_columns].isna().any(axis=1).sum())
    assert missing == 0, (
        f"{missing:,} rows have no weather -- the grid extends past the weather series. "
        f"Re-run 1.4, which cuts the taxi data at the weather cutoff."
    )

    # The join key just duplicates `timestamp`, so it is not a feature.
    grid_df = grid_df.drop(columns=[time_step_column])

    return grid_df

## 4. POI Features

In [ ]:
# Every grid file is named complete_grid_<spatial_level>_<temporal_level>, so the name alone tells us
# which spatial level a grid is on -- and the spatial level alone decides which POI table to merge
# and which key to merge it on. That is the whole registry.
POI_LEVELS = {
    'h3_7':      {'spatial_column': 'h3_index_7', 'pois': pois_cat_wide_7},
    'census':    {'spatial_column': 'geoid10',    'pois': pois_cat_wide_census},
    'community': {'spatial_column': 'commarea',   'pois': pois_cat_wide_comm},
}

# The POI files key the h3 level on a generic 'h3_index'. Rename it to the grid's own key once,
# here, so a single merge works unchanged at every level.
for level, spec in POI_LEVELS.items():
    poi_key = 'h3_index' if level.startswith('h3') else spec['spatial_column']
    spec['pois'] = spec['pois'].rename(columns={poi_key: spec['spatial_column']})


def split_grid_name(grid_name):
    """'complete_grid_h3_7_hourly' -> ('h3_7', 'hourly')"""
    spatial_level, temporal_level = grid_name.removeprefix('complete_grid_').rsplit('_', 1)
    return spatial_level, temporal_level


for grid_name in grid_dfs:
    spatial_level, temporal_level = split_grid_name(grid_name)
    spec = POI_LEVELS[spatial_level]
    print(f"{grid_name:32s} -> {spatial_level:10s} {temporal_level:7s} "
          f"merges POIs on '{spec['spatial_column']}' ({len(spec['pois'])} units)")

In [11]:
def add_poi_features(grid_df, grid_name):
    """Merge the POI table belonging to this grid's spatial level, resolved from the grid name."""
    spatial_level, _ = split_grid_name(grid_name)
    spec = POI_LEVELS[spatial_level]
    pois, spatial_column = spec['pois'], spec['spatial_column']

    grid_df = grid_df.merge(pois, on=spatial_column, how='left')

    # A unit with no POIs really does have zero of them, so a 0 fill is correct here -- but scope it
    # to the POI columns rather than calling grid_df.fillna(0) on the whole frame, which would put a
    # meaningless 0 into any other column that happened to be missing.
    poi_columns = [c for c in pois.columns if c.startswith('poi_cat_')]
    grid_df[poi_columns] = grid_df[poi_columns].fillna(0).astype(int)

    return grid_df

# Feature Engineering Loop

In [12]:
def engineer_features(grid_df, grid_name):
    """Add every feature block to one grid. The grid name resolves both the spatial and the
    temporal level, so nothing has to be passed in by hand."""
    spatial_level, temporal_level = split_grid_name(grid_name)
    temporal_column = 'timestamp'

    # Drop the leakage first (see LEAKY_COLUMNS above). Also keeps the frame narrow through the joins.
    grid_df = grid_df.drop(columns=LEAKY_COLUMNS)

    grid_df['distance_to_loop'] = new_haversine_distance_to_loop(grid_df['lat'], grid_df['lon'])
    grid_df = add_holiday_feature(grid_df, temporal_column)
    grid_df = temporal_feature_engineering(grid_df, temporal_column, temporal_level)
    grid_df = create_weather_features(grid_df, weather_df, temporal_column)
    grid_df = add_poi_features(grid_df, grid_name)
    return grid_df

## 5. Prediction Data Split for Train, Validation and Test
**Validation Strategy**: Use a robust **Temporal Split** (avoiding autokorrelation and data leakage from random splits) to evaluate out-of-sample predictive performance.

In [ ]:
def temporal_split(grid_df, grid_name):
    """Split one grid into train/val/test on time (50/20/30) and attach the training-set base demand.

    Both the spatial key and the base-demand profile come from the grid name, so this works for every
    level. A random split would leak: consecutive hours in the same cell are highly autocorrelated.
    """
    spatial_level, temporal_level = split_grid_name(grid_name)
    spatial_column = POI_LEVELS[spatial_level]['spatial_column']
    temporal_column = 'timestamp'

    unique_dates = pd.Series(grid_df[temporal_column].dt.date.unique()).sort_values().reset_index(drop=True)
    train_end = unique_dates.iloc[int(len(unique_dates) * 0.50)]
    val_end   = unique_dates.iloc[int(len(unique_dates) * 0.70)]

    dates = grid_df[temporal_column].dt.date
    df_train = grid_df[dates < train_end]
    df_val   = grid_df[(dates >= train_end) & (dates < val_end)]
    df_test  = grid_df[dates >= val_end]

    print(f"  train < {train_end} | val < {val_end} | "
          f"rows {len(df_train):,} / {len(df_val):,} / {len(df_test):,}")

    # Historical base demand, computed ONLY on train and applied to all three splits.
    keys = base_demand_keys(spatial_column, temporal_level)
    base = df_train.groupby(keys)["Total_Trip_Start"].mean().rename("base_demand").reset_index()
    global_base = df_train["Total_Trip_Start"].mean()
    print(f"  base_demand keyed on {keys}")

    return {
        'train': add_base_demand(df_train, base, keys, global_base),
        'val':   add_base_demand(df_val, base, keys, global_base),
        'test':  add_base_demand(df_test, base, keys, global_base),
    }


def export_grid_dfs(grid_name, grid_df, output_folder="../data/prediction_split/"):
    """Split `grid_df` and write it to prediction_split/<spatial_level>/<temporal_level>/<split>.parquet

    The directory path already names the spatial and temporal level, so the file only has to say
    which split it is -- repeating 'complete_grid_h3_7_hourly' in the filename adds nothing.

        data/prediction_split/h3_7/hourly/{train,val,test}.parquet
        data/prediction_split/census/daily/{train,val,test}.parquet
    """
    spatial_level, temporal_level = split_grid_name(grid_name)
    output_folder = Path(output_folder) / spatial_level / temporal_level
    output_folder.mkdir(parents=True, exist_ok=True)

    splits = temporal_split(grid_df, grid_name)
    for split_name, df in splits.items():
        df.to_parquet(output_folder / f"{split_name}.parquet", index=False)
    print(f"  exported train/val/test to {output_folder}")

In [ ]:
import gc

# Engineer, split and export ONE grid at a time.
#
# Feature engineering roughly quadruples a grid's width (14 raw columns -> ~44), so holding all six
# engineered grids at once needs several GB, and the train/val/test copies on top of that will not
# fit in 16 GB of RAM. Each grid is therefore reloaded from disk, processed, written and dropped
# before the next one starts, which keeps the peak at roughly one grid plus its splits.
del grid_dfs
gc.collect()

for grid_file in sorted(grid_files):
    grid_name = grid_file.stem
    spatial_level, temporal_level = split_grid_name(grid_name)
    print(f"{spatial_level} / {temporal_level}")

    grid_df = engineer_features(pd.read_parquet(grid_file), grid_name)
    print(f"  {grid_df.shape[0]:,} rows x {grid_df.shape[1]} columns after feature engineering")

    export_grid_dfs(grid_name, grid_df)

    del grid_df
    gc.collect()

print("\nAll grids engineered, split and exported.")